## **Train and Evaluate a Keras-Based Classifier**
### **Objective**
    1. Creating a Keras-based convolution neural network (CNN) model.
    2. Train the CNN model on agricultural and non-agricultural land dataset.
    3. Evaluate the performance of the CNN model.

##### **Import Libraries**

In [ ]:
import os
import sys
import time
import random
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import subprocess
import importlib
from pathlib import Path
import tarfile

: 

In [6]:
# 
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.initializers import HeUniform
from tensorflow.keras.callbacks import ModelCheckpoint

from sklearn.metrics import accuracy_score
print("Succesfully imported the libraries")

Succesfully imported the libraries


##### **Data download**

In [ ]:
# Extraction directory for the dataset
extract_dir = "C:\\Users\\jdmay\\AI-Projects\\Landuse-classification"

In [ ]:
# URL for the dataset
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4Z1fwRR295-1O3PMQBH6Dg/images-dataSAT.tar"

In [ ]:

def check_skillnetwork_extraction(extract_dir):
    """Check write permissions by testing symlink creation."""
    symlink_test = os.path.join(extract_dir, "symlink_test")
    if not os.path.exists(symlink_test):
        os.symlink(os.path.join(os.sep, "tmp"), symlink_test)
        print("Write permissions available for downloading and extracting the dataset tar file")
    os.unlink(symlink_test)


async def download_tar_dataset(url, tar_path, extract_dir):
    """Download url to tar_path then extract to extract_dir."""
    if not os.path.exists(tar_path):
        try:
            try:
                import httpx
            except ImportError:
                print("Installing httpx...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "httpx"])
                httpx = importlib.import_module("httpx")

            print(f"Downloading from {url}...")
            async with httpx.AsyncClient() as client:
                response = await client.get(url, follow_redirects=True)
                response.raise_for_status()

            with open(tar_path, "wb") as f:
                f.write(response.content)
            print(f"Successfully downloaded '{Path(tar_path).name}'.")
        except Exception as e:
            print(f"Download failed: {e}")
            raise
    else:
        print(f"Tar file already exists at: {tar_path}")

    with tarfile.open(tar_path, "r:*") as tar_ref:
        tar_ref.extractall(path=extract_dir)
    print(f"Successfully extracted to '{extract_dir}'.")


async def ensure_dataset(url, extract_dir):
    """Try skillsnetwork first; fall back to manual download/extract."""
    try:
        import skillsnetwork
        check_skillnetwork_extraction(extract_dir)
        await skillsnetwork.prepare(url=url, path=extract_dir, overwrite=True)
    except Exception as e:
        print(f"{e}\nPrimary method failed. Falling back to manual download...")
        tar_path = os.path.join(extract_dir, Path(url).name)
        await download_tar_dataset(url, tar_path, extract_dir)


await ensure_dataset(url, extract_dir)


Write permissions available for downloading and extracting the dataset tar file
[Errno 2] No such file or directory: '\\tmp\\skills-network-3594948816945861240-images-dataSAT.tar'
Primary method failed. Falling back to manual download...
Successfully downloaded 'images-dataSAT.tar'.
Successfully extracted to 'C:\Users\jdmay\AI-Projects\Landuse-classification'.


Check GPU availability

In [8]:
gpu_list = tf.config.list_physical_devices('GPU')

device = 'gpu' if gpu_list !=[] else 'cpu'
print(f'Device available for training: {device}')

Device available for training: cpu
